# Results

Charts the measured benchmark results. Every number is read from
`benchmarks/results.json`, which is written by the benchmark scripts — so this
notebook can never drift from what was actually measured.

Regenerate the underlying numbers with:

```
python benchmarks/run_dnn.py
python benchmarks/run_openai_prompted.py
python benchmarks/run_groq.py
```

In [ ]:
import json
from pathlib import Path

import plotly.graph_objects as go

results = json.loads(Path("benchmarks/results.json").read_text(encoding="utf-8"))
results.sort(key=lambda r: r["avg_error"])

for r in results:
    print(f"{r['model']:<28} ${r['avg_error']:>7.2f}   {r['hit_rate_pct']:>5.1f}% within 20%   n={r['n']}")

In [ ]:
# Average absolute error - lower is better. Best result highlighted.
labels = [r["model"] for r in results]
values = [r["avg_error"] for r in results]
colors = ["#3fcf8e"] + ["#7c6cff"] * (len(results) - 1)

fig = go.Figure(
    go.Bar(
        x=labels,
        y=values,
        marker_color=colors,
        text=[f"${v:,.2f}" for v in values],
        textposition="outside",
    )
)
fig.update_layout(
    title=f"Average absolute error on {results[0]['n']} held-out products (lower is better)",
    yaxis=dict(title="Average absolute error ($)", range=[0, max(values) * 1.18]),
    xaxis=dict(tickangle=-20),
    template="plotly_white",
    width=900,
    height=520,
    showlegend=False,
)
fig.show()

In [ ]:
# Accuracy framed as hit rate: how often the estimate lands within 20% of the truth.
hits = [r["hit_rate_pct"] for r in results]

fig = go.Figure(
    go.Bar(
        x=labels,
        y=hits,
        marker_color=colors,
        text=[f"{h:.1f}%" for h in hits],
        textposition="outside",
    )
)
fig.update_layout(
    title="Share of estimates within 20% of the actual price (higher is better)",
    yaxis=dict(title="Within 20% (%)", range=[0, 100]),
    xaxis=dict(tickangle=-20),
    template="plotly_white",
    width=900,
    height=520,
    showlegend=False,
)
fig.show()

## A note on the fine-tuned model

An earlier version of this project also compared a fine-tuned `gpt-4.1-nano`.
That row cannot be reproduced any more.

OpenAI wound down its self-serve fine-tuning platform on **7 May 2026**.
Organisations that had not already run a fine-tuning job lost the ability to
create one, and the API now returns:

```
403 - OpenAI is winding down the fine-tuning platform and your organization
is no longer able to create new fine-tuning training jobs.
code: training_not_available
```

`benchmarks/run_finetune.py` is kept in the repository so the attempt — and its
failure — stays reproducible. See OpenAI's
[deprecations page](https://developers.openai.com/api/docs/deprecations).